# YouTube Sports Predictions

- **Question:** Which sport analyst is correct with the team they predict to win?
- **Tasks:**
    
    1. Find any pre-game YouTube clip where the sport analysts make predictions of which team they think will win.
        - Ex. See [Immediate 2025 NCAA tournament Final Four and championship picks](https://youtu.be/-rjnvL9LL3U?si=QMFJYAQ8Q85lTNCD)
    2. Find a post-game YouTube clip where the winning team (from 1) is announced.
        - Ex. See [Florida vs. Houston - 2025 NCAA men’s national championship | FULL REPLAY](https://www.youtube.com/watch?v=UHJ90jJmCYw)
    3. Use the [`youtube-transcript-api`](https://pypi.org/project/youtube-transcript-api/) to retrieve the transcript of both videos (from 1 and 2).
    4. Extract the transcripts
    5. Clean the data
    6. Create prompt to identify predictions.
    7. Pass each row from video 1 + prompt into the LLMs.
    8. Majority Vote (MV) with LLMs
    9. Filter by MV label = 1, hence prediction
    10. For each prediction, compare with each observation sentence from video 2. 
    11. Prompt LLMs to identify if observation from video 2 is enough to state if each prediction came true.
- **Misc:**
    
    1. Watch the video first to ensure it contains predictions (the team that is predicted to win) and observations (the team that wins)
    2. Shorten the transcript to only have predictions/observations
    3. Could focus soley on finding predictions only, so can skip 2, 10, 11. 

In [33]:
import os
import sys

import pandas as pd

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Preprocessing code

- Will use the below class and functions for both video transcripts.

---

1. `YouTubeTranscriptApi()` to fetch the YT transcripts.
2. `extract_data()` to transform data into Pandas DataFrame.
3. `clean_data()` to properly form sentences.

In [3]:
ytt_api = YouTubeTranscriptApi()

In [4]:
def extract_data(snippets) -> pd.DataFrame:
    """Extract the text, start time, and duration from the YT `FetchedTranscriptSnippet()`."""
    snippet_texts = []
    snippet_starts = []
    snippet_durations = []
    data = {}

    for snippets_idx in range(len(snippets)):
        snippet = snippets[snippets_idx]
        snippet_texts.append(snippet.text)
        snippet_starts.append(snippet.start)
        snippet_durations.append(snippet.duration)

    data['Text'] = snippet_texts
    data['Start Time'] = snippet_starts
    data['Duration'] = snippet_durations

    df = pd.DataFrame(data=data)
    return df

In [5]:
def clean_data(df) -> list[str]:
    """Rows can contain multiple sentences and some sentences can be on multiple rows, so ensure we 
    join proper sentences together.
    """

    text = df.Text.to_list()
    text_joined = ' '.join(text)
    # print(f"{text_joined}")
    text_split = text_joined.split('.')
    return text_split

## Pre-Game Clip

In [ ]:
# FULL PREVIEW & PICKS: Patriots vs. Seahawks Super Bowl LX 🏆 Who wins the Lombardi Trophy? | NFL Live
# "https://www.youtube.com/watch?v=ZZN7BAYeOtc

# FIRST TAKE'S SUPER BOWL PICKS! The crew is going with... 😱
# https://www.youtube.com/watch?v=mBK8o5orBbE

# NFL Predictions and Picks For Super Bowl LX [Patriots vs Seahawks] | Best Bets ✅
# https://www.youtube.com/watch?v=LXPQrZV4Cfw

# Rich Eisen’s Pick to Win the Seahawks vs Patriots Super Bowl LX Is….? | The Rich Eisen Show
# https://www.youtube.com/watch?v=fUmJAtFEGn8

# The Pat McAfee Show's Picks For Super Bowl LX
# https://www.youtube.com/watch?v=MTVAkVkkaz4

# Super Bowl LX On-Site Preview: Picks, Predictions, Everything you need to know for Patriots-Seahawks
# https://www.youtube.com/watch?v=Z0xP3GNpjkw

video_id = "Z0xP3GNpjkw"
pre_game_snippets = ytt_api.fetch(video_id)
pre_game_snippets[:7]

[FetchedTranscriptSnippet(text='episodes now strmingnlyn', start=0.867, duration=10.073),
 FetchedTranscriptSnippet(text='Paraunt+ Welme t CBS', start=2.868, duration=9.206),
 FetchedTranscriptSnippet(text='orts HQ,reseed b Han.Itas t beas', start=7.171, duration=6.704),
 FetchedTranscriptSnippet(text='s toe a gooshowecau', start=11.074, duration=3.969),
 FetchedTranscriptSnippet(text="it's theridaof Ser Bow", start=12.174, duration=4.67),
 FetchedTranscriptSnippet(text="weekI'm nny Dl, t Sup", start=13.975, duration=4.07),
 FetchedTranscriptSnippet(text='Bowl theay aer torro We', start=15.143, duration=4.636)]

In [52]:
pre_game_snippets_df = extract_data(pre_game_snippets)
pre_game_snippets_df

,Text,Start Time,Duration
0,episodes now strmingnlyn,0.867,10.073
1,Paraunt+ Welme t CBS,2.868,9.206
2,"orts HQ,reseed b Han.Itas t beas",7.171,6.704
3,s toe a gooshowecau,11.074,3.969
4,it's theridaof Ser Bow,12.174,4.670
...,...,...,...
371,waveng. m no waving.,742.411,3.902
372,Fell. Thbestart abo,743.411,5.971
373,beinthe int gua is just,744.512,5.170
374,I ju disit outI dotne to ke aicu,746.413,3.269


In [53]:
pre_game_snippets_df['Video ID'] = video_id
pre_game_snippets_df

,Text,Start Time,Duration,Video ID
0,episodes now strmingnlyn,0.867,10.073,Z0xP3GNpjkw
1,Paraunt+ Welme t CBS,2.868,9.206,Z0xP3GNpjkw
2,"orts HQ,reseed b Han.Itas t beas",7.171,6.704,Z0xP3GNpjkw
3,s toe a gooshowecau,11.074,3.969,Z0xP3GNpjkw
4,it's theridaof Ser Bow,12.174,4.670,Z0xP3GNpjkw
...,...,...,...,...
371,waveng. m no waving.,742.411,3.902,Z0xP3GNpjkw
372,Fell. Thbestart abo,743.411,5.971,Z0xP3GNpjkw
373,beinthe int gua is just,744.512,5.170,Z0xP3GNpjkw
374,I ju disit outI dotne to ke aicu,746.413,3.269,Z0xP3GNpjkw


In [54]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "nfl_sb_lx", "pre_game")
DataProcessing.save_to_file(pre_game_snippets_df, path=save_data_path, prefix=f'{video_id}', save_file_type='csv')

Using file number: 1
Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/nfl_sb_lx/pre_game/Z0xP3GNpjkw-v1.csv
